# StyleFormer Inference on Kaggle

Run face transformation inference with trained models.

## Prerequisites
- Add datasets:
  - `styleformer` (codebase)
  - Your trained model checkpoint OR pretrained weights
  - Test images (or use sample from CelebA)

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q lpips einops pytorch-lightning

In [ ]:
import sys
import os
from pathlib import Path

# Add StyleFormer to path
STYLEFORMER_PATH = Path('/kaggle/input/styleformer')
if not STYLEFORMER_PATH.exists():
    for alt in ['/kaggle/input/styleformer-code', '/kaggle/working/StyleFormer']:
        if Path(alt).exists():
            STYLEFORMER_PATH = Path(alt)
            break

sys.path.insert(0, str(STYLEFORMER_PATH))

from kaggle.setup_kaggle import setup, KaggleConfig
config = KaggleConfig()
setup(install_packages=False)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

DEVICE = config.get_device()
print(f"Using device: {DEVICE}")

## 2. Load Model

In [ ]:
from src.inference import InferencePipeline
from src.data.transforms import get_inference_transforms, denormalize

# Check for available checkpoints
checkpoint_paths = [
    '/kaggle/input/styleformer-weights/model.ckpt',
    '/kaggle/working/checkpoints/final_model.ckpt',
    '/kaggle/working/outputs/final_model.ckpt',
]

CHECKPOINT_PATH = None
for path in checkpoint_paths:
    if Path(path).exists():
        CHECKPOINT_PATH = path
        break

if CHECKPOINT_PATH:
    print(f"Found checkpoint: {CHECKPOINT_PATH}")
else:
    print("No checkpoint found. Using demo mode with random weights.")

In [ ]:
# Define model architecture (must match training)
from src.models.encoders.base import BaseEncoder
from src.models.lightning_modules.base import BaseTransformerModule


class SimpleEncoder(BaseEncoder):
    """Simple encoder matching training architecture."""
    
    def __init__(self, w_dim=512, num_ws=14, input_size=256):
        super().__init__(w_dim=w_dim, num_ws=num_ws, input_size=input_size)
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 3, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        
        self.styles = nn.ModuleList([
            nn.Linear(512, w_dim) for _ in range(num_ws)
        ])
    
    def forward(self, x):
        features = self.backbone(x).flatten(1)
        styles = [layer(features) for layer in self.styles]
        return torch.stack(styles, dim=1)


class SimpleDecoder(nn.Module):
    """Simple decoder matching training architecture."""
    
    def __init__(self, w_dim=512, output_size=256):
        super().__init__()
        self.const = nn.Parameter(torch.randn(1, 512, 4, 4))
        
        self.layers = nn.ModuleList([
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ConvTranspose2d(64, 64, 4, 2, 1),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ConvTranspose2d(32, 16, 4, 2, 1),
        ])
        
        self.to_rgb = nn.Conv2d(16, 3, 1)
        self.act = nn.LeakyReLU(0.2)
    
    def forward(self, w):
        x = self.const.repeat(w.shape[0], 1, 1, 1)
        for layer in self.layers:
            x = self.act(layer(x))
        return torch.tanh(self.to_rgb(x))


# Create model
encoder = SimpleEncoder().to(DEVICE)
decoder = SimpleDecoder().to(DEVICE)

# Load weights if available
if CHECKPOINT_PATH:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    if 'encoder' in checkpoint:
        encoder.load_state_dict(checkpoint['encoder'])
        decoder.load_state_dict(checkpoint['decoder'])
    elif 'state_dict' in checkpoint:
        # PyTorch Lightning format
        state_dict = checkpoint['state_dict']
        encoder_dict = {k.replace('encoder.', ''): v for k, v in state_dict.items() if k.startswith('encoder.')}
        decoder_dict = {k.replace('decoder.', ''): v for k, v in state_dict.items() if k.startswith('decoder.')}
        encoder.load_state_dict(encoder_dict)
        decoder.load_state_dict(decoder_dict)
    print("Loaded model weights!")

encoder.eval()
decoder.eval()
print("Model ready for inference!")

## 3. Image Preprocessing

In [ ]:
from torchvision import transforms

# Preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # [-1, 1]
])

def load_image(path):
    """Load and preprocess an image."""
    img = Image.open(path).convert('RGB')
    return preprocess(img).unsqueeze(0).to(DEVICE)

def tensor_to_image(tensor):
    """Convert tensor to displayable image."""
    img = tensor.squeeze(0).cpu()
    img = (img + 1) / 2  # [-1, 1] -> [0, 1]
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)

## 4. Run Inference

In [ ]:
# Find test images
test_image_paths = list(Path('/kaggle/input').rglob('*.jpg'))[:10]
if not test_image_paths:
    test_image_paths = list(Path('/kaggle/input').rglob('*.png'))[:10]

print(f"Found {len(test_image_paths)} test images")
if test_image_paths:
    print(f"First image: {test_image_paths[0]}")

In [ ]:
# Process images
if test_image_paths:
    results = []
    
    for path in test_image_paths[:8]:
        try:
            # Load and process
            img_tensor = load_image(path)
            
            with torch.no_grad():
                # Encode to latent
                latent = encoder(img_tensor)
                
                # Decode (reconstruction)
                reconstructed = decoder(latent)
            
            results.append({
                'path': path,
                'original': img_tensor,
                'latent': latent,
                'reconstructed': reconstructed,
            })
        except Exception as e:
            print(f"Error processing {path}: {e}")
    
    print(f"Processed {len(results)} images")

In [ ]:
# Visualize results
if results:
    n = min(len(results), 4)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    
    for i in range(n):
        # Original
        axes[0, i].imshow(tensor_to_image(results[i]['original']))
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')
        
        # Reconstructed
        axes[1, i].imshow(tensor_to_image(results[i]['reconstructed']))
        axes[1, i].set_title('Reconstructed')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/inference_results.png', dpi=150)
    plt.show()

## 5. Latent Space Manipulation (if directions available)

In [ ]:
# Check for attribute directions
direction_paths = [
    '/kaggle/input/styleformer/data/directions',
    '/kaggle/input/interfacegan-directions',
]

DIRECTIONS = {}
for dir_path in direction_paths:
    p = Path(dir_path)
    if p.exists():
        for f in p.glob('*.pt'):
            name = f.stem.replace('_direction', '')
            DIRECTIONS[name] = torch.load(f, map_location=DEVICE)
            print(f"Loaded direction: {name}")

if not DIRECTIONS:
    print("No attribute directions found. Creating random directions for demo.")
    DIRECTIONS = {
        'age': torch.randn(512, device=DEVICE) * 0.1,
        'smile': torch.randn(512, device=DEVICE) * 0.1,
    }

In [ ]:
# Latent manipulation demo
if results and DIRECTIONS:
    img_tensor = results[0]['original']
    base_latent = results[0]['latent']
    
    # Pick an attribute
    attr_name = list(DIRECTIONS.keys())[0]
    direction = DIRECTIONS[attr_name]
    
    # Expand direction to match latent shape
    if direction.dim() == 1:
        direction = direction.unsqueeze(0).unsqueeze(0)  # (1, 1, 512)
    
    # Generate interpolation
    strengths = [-3, -2, -1, 0, 1, 2, 3]
    interpolated = []
    
    with torch.no_grad():
        for s in strengths:
            edited_latent = base_latent + s * direction
            edited_image = decoder(edited_latent)
            interpolated.append(tensor_to_image(edited_image))
    
    # Plot
    fig, axes = plt.subplots(1, len(strengths), figsize=(3*len(strengths), 3))
    for i, (img, s) in enumerate(zip(interpolated, strengths)):
        axes[i].imshow(img)
        axes[i].set_title(f'{attr_name}={s}')
        axes[i].axis('off')
    
    plt.suptitle(f'Latent Interpolation: {attr_name}')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{attr_name}_interpolation.png', dpi=150)
    plt.show()

## 6. Batch Processing

In [ ]:
def process_directory(input_dir, output_dir, batch_size=8):
    """Process all images in a directory."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all images
    image_paths = list(input_dir.glob('*.jpg')) + list(input_dir.glob('*.png'))
    print(f"Found {len(image_paths)} images")
    
    # Process in batches
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        
        # Load batch
        batch = torch.cat([load_image(p) for p in batch_paths], dim=0)
        
        with torch.no_grad():
            latents = encoder(batch)
            outputs = decoder(latents)
        
        # Save outputs
        for j, path in enumerate(batch_paths):
            output_path = output_dir / path.name
            img = tensor_to_image(outputs[j:j+1])
            Image.fromarray((img * 255).astype(np.uint8)).save(output_path)
        
        print(f"Processed batch {i//batch_size + 1}/{(len(image_paths)-1)//batch_size + 1}")
    
    return len(image_paths)

# Example usage:
# process_directory('/kaggle/input/my-images', '/kaggle/working/outputs')

## 7. Save Results

In [ ]:
# Save all generated images
output_dir = Path('/kaggle/working/generated')
output_dir.mkdir(exist_ok=True)

if results:
    for i, result in enumerate(results):
        # Save reconstructed
        img = tensor_to_image(result['reconstructed'])
        Image.fromarray((img * 255).astype(np.uint8)).save(
            output_dir / f'reconstructed_{i:03d}.png'
        )
    print(f"Saved {len(results)} images to {output_dir}")

In [ ]:
# List all outputs
!ls -la /kaggle/working/